# 04 – Model Training
**AlchemiX – Launch26 Phase 2**

## Mandatory Workflow
```
Load Engineered Data
       ↓
Train / Test Split   ← MUST happen FIRST
       ↓
Historical Feature Engineering (on training set ONLY, mapped to test)
       ↓
Encoding (OrdinalEncoder)
       ↓
Scaling (StandardScaler)
       ↓
Benchmark Models
       ↓
Hyperparameter Tuning (best model)
       ↓
Save Best Models
```

> **Critical rule:** Historical statistics (mean, std, attack_rate) are computed
> on the training set ONLY and then mapped to the test set. They are NEVER
> computed on the full dataset before splitting.

In [1]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from ml.utils import (
    TRAFFIC_ENG_FILE, TELEMETRY_ENG_FILE, INCIDENT_ENG_FILE,
    CONGESTION_MODEL_FILE, TRUST_MODEL_FILE, TARGETING_MODEL_FILE,
    RANDOM_SEED,
)
from ml.training import (
    split_data,
    add_historical_features_traffic,
    add_historical_features_telemetry,
    add_historical_features_incident,
    regression_candidates,
    classification_candidates,
    benchmark_models,
    tune_model,
    build_preprocessor,
)
from ml.evaluation import regression_metrics, classification_metrics, print_comparison_table

np.random.seed(RANDOM_SEED)
print('Imports OK ✓')

Imports OK ✓


---
## ① Congestion Model (Traffic → observed_latency_ms)

In [2]:
traffic_eng = pd.read_csv(TRAFFIC_ENG_FILE)
print(f'Traffic engineered: {traffic_eng.shape}')
display(traffic_eng.head(2))

Traffic engineered: (5743, 15)


,link_id,tick,load_units,load_ratio,status,observed_latency_ms,planet_a,planet_b,load_percentage,near_capacity,high_congestion,load_bucket,status_ok,load_ratio_sq,load_units_x_ratio
0,Aegis-Boreas,0,90.0,0.4328,ok,118635.583,Aegis,Boreas,43.28,0,0,medium,1,0.187316,38.95200
1,Aegis-Boreas,1,136.4,0.6557,ok,283818.179,Aegis,Boreas,65.57,0,0,high,1,0.429942,89.43748


In [3]:
# ── 1. Train/Test Split ──────────────────────────────────────────────────────
TARGET_T = 'observed_latency_ms'
train_t, test_t = split_data(traffic_eng, target_col=TARGET_T)
print(f'Train: {len(train_t)}  Test: {len(test_t)}')

03:29:29  INFO      ml.training  split_data: train=4594  test=1149  target='observed_latency_ms'


Train: 4594  Test: 1149


In [4]:
# ── 2. Historical features AFTER split ──────────────────────────────────────
train_t, test_t = add_historical_features_traffic(train_t, test_t)
print('Historical features added. New columns:', [c for c in train_t.columns if 'hist_' in c])

03:29:29  INFO      ml.training  add_historical_features_traffic: added 3 historical columns


Historical features added. New columns: ['hist_mean_load_ratio', 'hist_std_load_ratio', 'hist_mean_load_units']


In [5]:
# ── 3. Define feature sets ───────────────────────────────────────────────────
TRAFFIC_NUMERIC = [
    'load_units', 'load_ratio', 'load_percentage', 'load_ratio_sq',
    'load_units_x_ratio', 'near_capacity', 'high_congestion', 'status_ok',
    'hist_mean_load_ratio', 'hist_std_load_ratio', 'hist_mean_load_units',
]
TRAFFIC_CATEGORICAL = ['planet_a', 'planet_b']

# Verify no target leakage
assert TARGET_T not in TRAFFIC_NUMERIC + TRAFFIC_CATEGORICAL, 'Target in features!'

X_train_t = train_t[TRAFFIC_NUMERIC + TRAFFIC_CATEGORICAL]
y_train_t = train_t[TARGET_T].values
X_test_t  = test_t[TRAFFIC_NUMERIC + TRAFFIC_CATEGORICAL]
y_test_t  = test_t[TARGET_T].values

print(f'X_train: {X_train_t.shape}  X_test: {X_test_t.shape}')

X_train: (4594, 13)  X_test: (1149, 13)


In [6]:
# ── 4. Preprocessing pipeline ────────────────────────────────────────────────
preprocessor_t = build_preprocessor(TRAFFIC_NUMERIC, TRAFFIC_CATEGORICAL)
X_train_t_proc = preprocessor_t.fit_transform(X_train_t)
X_test_t_proc  = preprocessor_t.transform(X_test_t)
print(f'Preprocessed X_train: {X_train_t_proc.shape}')

Preprocessed X_train: (4594, 13)


In [7]:
# ── 5. Benchmark ─────────────────────────────────────────────────────────────
print('Benchmarking congestion models...')
candidates_t = regression_candidates(RANDOM_SEED)
results_t = benchmark_models(candidates_t, X_train_t_proc, y_train_t, task='regression')
print_comparison_table(results_t)

Benchmarking congestion models...


03:29:33  INFO      ml.training    RandomForest     cv_mean=-56005.5619  cv_std=30893.4232  time=3.6s
03:29:37  INFO      ml.training    XGBoost          cv_mean=-65838.5893  cv_std=34848.9846  time=4.0s
03:29:40  INFO      ml.training    LightGBM         cv_mean=-58681.8740  cv_std=32294.6744  time=2.7s
03:29:42  INFO      ml.training    DecisionTree     cv_mean=-61124.0440  cv_std=27572.6730  time=2.0s



  MODEL COMPARISON
       model       cv_mean       cv_std  train_time_s
     XGBoost -65838.589264 34848.984638          4.02
DecisionTree -61124.043964 27572.672981          2.04
    LightGBM -58681.873967 32294.674363          2.67
RandomForest -56005.561910 30893.423180          3.58



In [8]:
# ── 6. Tune best model ───────────────────────────────────────────────────────
best_name_t = results_t.iloc[0]['model']
best_model_t_raw = regression_candidates(RANDOM_SEED)[best_name_t]
print(f'Best model: {best_name_t} – tuning...')
congestion_model = tune_model(
    best_model_t_raw, best_name_t, X_train_t_proc, y_train_t, task='regression'
)
congestion_model.fit(X_train_t_proc, y_train_t)

Best model: XGBoost – tuning...


03:29:50  INFO      ml.training  tune_model [XGBoost]: best score=-58992.3631  params={'subsample': 0.8, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 1.0}


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,1.0
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [9]:
# ── 7. Test set evaluation ───────────────────────────────────────────────────
y_pred_t = congestion_model.predict(X_test_t_proc)
print('Congestion Model – Test Set Metrics:')
metrics_t = regression_metrics(y_test_t, y_pred_t)
pd.DataFrame([metrics_t])

03:29:50  INFO      ml.evaluation    MAE        = 13870.0540
03:29:50  INFO      ml.evaluation    RMSE       = 54646.3550
03:29:50  INFO      ml.evaluation    R2         = 0.8917
03:29:50  INFO      ml.evaluation    MAPE_%     = 8.1580


Congestion Model – Test Set Metrics:


,MAE,RMSE,R2,MAPE_%
0,13870.054016,54646.354997,0.89168,8.15796


---
## ② Trust Model (Telemetry → trust_score)

In [10]:
telemetry_eng = pd.read_csv(TELEMETRY_ENG_FILE)
print(f'Telemetry engineered: {telemetry_eng.shape}')

Telemetry engineered: (6000, 11)


In [11]:
TARGET_TE = 'trust_score'
train_te, test_te = split_data(telemetry_eng, target_col=TARGET_TE)
train_te, test_te = add_historical_features_telemetry(train_te, test_te)
print(f'Train: {len(train_te)}  Test: {len(test_te)}')

03:29:50  INFO      ml.training  split_data: train=4800  test=1200  target='trust_score'
03:29:51  INFO      ml.training  add_historical_features_telemetry: added 3 historical columns


Train: 4800  Test: 1200


In [12]:
TELE_NUMERIC = [
    'self_reported_latency_ms', 'measured_latency_ms',
    'latency_difference', 'absolute_difference', 'latency_ratio',
    'percentage_error',
    'hist_mean_trust', 'hist_std_trust', 'hist_mean_latency_diff',
]
TELE_CATEGORICAL = ['planet_a', 'planet_b']

assert TARGET_TE not in TELE_NUMERIC + TELE_CATEGORICAL, 'Target in features!'

X_train_te = train_te[TELE_NUMERIC + TELE_CATEGORICAL]
y_train_te = train_te[TARGET_TE].values
X_test_te  = test_te[TELE_NUMERIC + TELE_CATEGORICAL]
y_test_te  = test_te[TARGET_TE].values

preprocessor_te = build_preprocessor(TELE_NUMERIC, TELE_CATEGORICAL)
X_train_te_proc = preprocessor_te.fit_transform(X_train_te)
X_test_te_proc  = preprocessor_te.transform(X_test_te)
print(f'X_train: {X_train_te_proc.shape}')

X_train: (4800, 11)


In [13]:
print('Benchmarking trust models...')
candidates_te = regression_candidates(RANDOM_SEED)
results_te = benchmark_models(candidates_te, X_train_te_proc, y_train_te, task='regression')
print_comparison_table(results_te)

Benchmarking trust models...


03:29:51  INFO      ml.training    RandomForest     cv_mean=-0.0006  cv_std=0.0002  time=0.8s
03:29:51  INFO      ml.training    XGBoost          cv_mean=-0.0013  cv_std=0.0003  time=0.1s
03:29:52  INFO      ml.training    LightGBM         cv_mean=-0.0019  cv_std=0.0005  time=0.5s
03:29:52  INFO      ml.training    DecisionTree     cv_mean=-0.0007  cv_std=0.0001  time=0.1s



  MODEL COMPARISON
       model   cv_mean   cv_std  train_time_s
    LightGBM -0.001860 0.000491          0.46
     XGBoost -0.001298 0.000292          0.07
DecisionTree -0.000717 0.000122          0.13
RandomForest -0.000598 0.000222          0.83



In [14]:
best_name_te = results_te.iloc[0]['model']
best_model_te_raw = regression_candidates(RANDOM_SEED)[best_name_te]
print(f'Best model: {best_name_te} – tuning...')
trust_model = tune_model(
    best_model_te_raw, best_name_te, X_train_te_proc, y_train_te, task='regression'
)
trust_model.fit(X_train_te_proc, y_train_te)

y_pred_te = trust_model.predict(X_test_te_proc)
print('Trust Model – Test Set Metrics:')
metrics_te = regression_metrics(y_test_te, y_pred_te)

Best model: LightGBM – tuning...


03:30:24  INFO      ml.training  tune_model [LightGBM]: best score=-0.0015  params={'subsample': 0.8, 'num_leaves': 31, 'n_estimators': 300, 'learning_rate': 0.1}
03:30:24  INFO      ml.evaluation    MAE        = 0.0003
03:30:24  INFO      ml.evaluation    RMSE       = 0.0009
03:30:24  INFO      ml.evaluation    R2         = 0.9999
03:30:24  INFO      ml.evaluation    MAPE_%     = 0.0375


Trust Model – Test Set Metrics:


---
## ③ Targeting Model (Incident → jammed_flag)

In [15]:
incident_eng = pd.read_csv(INCIDENT_ENG_FILE)
incident_eng['jammed_flag'] = incident_eng['jammed_flag'].astype(int)
print(f'Incident engineered: {incident_eng.shape}')
print(f'Jam rate: {incident_eng["jammed_flag"].mean():.3f}')

Incident engineered: (6000, 11)
Jam rate: 0.082


In [16]:
TARGET_I = 'jammed_flag'
train_i, test_i = split_data(incident_eng, target_col=TARGET_I, stratify_col=TARGET_I)
train_i, test_i = add_historical_features_incident(train_i, test_i)
print(f'Train: {len(train_i)}  Test: {len(test_i)}')
print(f'Train jam rate: {train_i[TARGET_I].mean():.3f}  Test jam rate: {test_i[TARGET_I].mean():.3f}')

03:30:24  INFO      ml.training  split_data: train=4800  test=1200  target='jammed_flag'
03:30:24  INFO      ml.training  add_historical_features_incident: added 1 historical column


Train: 4800  Test: 1200
Train jam rate: 0.082  Test jam rate: 0.083


In [17]:
INC_NUMERIC = [
    'traffic_share', 'traffic_percentage', 'high_traffic_share',
    'relative_traffic', 'hist_attack_rate',
]
INC_CATEGORICAL = ['planet_a', 'planet_b']

assert TARGET_I not in INC_NUMERIC + INC_CATEGORICAL, 'Target in features!'

X_train_i = train_i[INC_NUMERIC + INC_CATEGORICAL]
y_train_i = train_i[TARGET_I].values
X_test_i  = test_i[INC_NUMERIC + INC_CATEGORICAL]
y_test_i  = test_i[TARGET_I].values

preprocessor_i = build_preprocessor(INC_NUMERIC, INC_CATEGORICAL)
X_train_i_proc = preprocessor_i.fit_transform(X_train_i)
X_test_i_proc  = preprocessor_i.transform(X_test_i)
print(f'X_train: {X_train_i_proc.shape}')

X_train: (4800, 7)


In [18]:
print('Benchmarking targeting models...')
candidates_i = classification_candidates(RANDOM_SEED)
results_i = benchmark_models(candidates_i, X_train_i_proc, y_train_i, task='classification')
print_comparison_table(results_i)

Benchmarking targeting models...


03:30:25  INFO      ml.training    RandomForest     cv_mean=0.5530  cv_std=0.0260  time=0.5s
03:30:25  INFO      ml.training    XGBoost          cv_mean=0.5740  cv_std=0.0355  time=0.1s
03:30:25  INFO      ml.training    LightGBM         cv_mean=0.5796  cv_std=0.0312  time=0.3s
03:30:25  INFO      ml.training    DecisionTree     cv_mean=0.5098  cv_std=0.0164  time=0.1s



  MODEL COMPARISON
       model  cv_mean   cv_std  train_time_s
    LightGBM 0.579596 0.031202          0.32
     XGBoost 0.574024 0.035463          0.08
RandomForest 0.552975 0.025974          0.47
DecisionTree 0.509766 0.016442          0.12



In [19]:
best_name_i = results_i.iloc[-1]['model']  # highest AUC (sorted ascending=False)
best_model_i_raw = classification_candidates(RANDOM_SEED)[best_name_i]
print(f'Best model: {best_name_i} – tuning...')
targeting_model = tune_model(
    best_model_i_raw, best_name_i, X_train_i_proc, y_train_i, task='classification'
)
targeting_model.fit(X_train_i_proc, y_train_i)

y_pred_i  = targeting_model.predict(X_test_i_proc)
y_prob_i  = targeting_model.predict_proba(X_test_i_proc)[:, 1]
print('Targeting Model – Test Set Metrics:')
metrics_i = classification_metrics(y_test_i, y_pred_i, y_prob_i)

Best model: DecisionTree – tuning...


03:30:26  INFO      ml.training  tune_model [DecisionTree]: best score=0.6006  params={'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 5}
03:30:26  INFO      ml.evaluation    Accuracy     = 0.9133
03:30:26  INFO      ml.evaluation    Precision    = 0.0000
03:30:26  INFO      ml.evaluation    Recall       = 0.0000
03:30:26  INFO      ml.evaluation    F1           = 0.0000
03:30:26  INFO      ml.evaluation    ROC_AUC      = 0.5847


Targeting Model – Test Set Metrics:


## Save Models & Preprocessors

In [20]:
import joblib

# Save models
joblib.dump(congestion_model, CONGESTION_MODEL_FILE)
joblib.dump(trust_model,      TRUST_MODEL_FILE)
joblib.dump(targeting_model,  TARGETING_MODEL_FILE)

# Save preprocessors for inference
from ml.utils import MODELS_DIR
joblib.dump(preprocessor_t,  MODELS_DIR / 'preprocessor_congestion.joblib')
joblib.dump(preprocessor_te, MODELS_DIR / 'preprocessor_trust.joblib')
joblib.dump(preprocessor_i,  MODELS_DIR / 'preprocessor_targeting.joblib')

# Save feature lists
import json
meta = {
    'congestion': {'numeric': TRAFFIC_NUMERIC, 'categorical': TRAFFIC_CATEGORICAL, 'target': TARGET_T},
    'trust':      {'numeric': TELE_NUMERIC, 'categorical': TELE_CATEGORICAL, 'target': TARGET_TE},
    'targeting':  {'numeric': INC_NUMERIC, 'categorical': INC_CATEGORICAL, 'target': TARGET_I},
}
(MODELS_DIR / 'model_metadata.json').write_text(json.dumps(meta, indent=2))

print('Models saved:')
print(f'  {CONGESTION_MODEL_FILE}')
print(f'  {TRUST_MODEL_FILE}')
print(f'  {TARGETING_MODEL_FILE}')
print('\n✓ Notebook 04 complete – proceed to Notebook 05 Evaluation.')

Models saved:
  E:\AlchemiX\models\congestion.joblib
  E:\AlchemiX\models\trust.joblib
  E:\AlchemiX\models\targeting.joblib

✓ Notebook 04 complete – proceed to Notebook 05 Evaluation.
